# Cell 1: Imports

In [1]:
import sagemaker
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    InputData, 
    S3DataSource, 
    SourceCode, 
    Compute,
    OutputDataConfig,
    StoppingCondition,
    MetricDefinition,
    StoppingCondition
)

[02/04/26 11:43:01] INFO     Found credentials in environment variables.                        ]8;id=923934;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=752700;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# --- USER SETTINGS ---
my_job_name = "3DCNN-Attention-Exp-1st_1000_resume" 
my_bucket = "alexander-thesis-cslr"

In [3]:
# 1. Metric Definitions (The Magic Part) 🔍
# Adjust the 'Regex' to match EXACTLY what your train_sagemaker.py prints.
# Example assumption: Your script prints "Epoch: 1, Loss: 0.45, Acc: 0.82"
# Updated Metric Definitions (using r'' to fix SyntaxWarning)
metrics = [
    MetricDefinition(name='train:loss',  regex=r'\[Metrics\] Train Loss: ([0-9\.]+)'),
    MetricDefinition(name='train:top1',  regex=r'\[Metrics\] Train Top1: ([0-9\.]+)'),
    MetricDefinition(name='train:top5',  regex=r'\[Metrics\] Train Top5: ([0-9\.]+)'),
    MetricDefinition(name='train:top10', regex=r'\[Metrics\] Train Top10: ([0-9\.]+)'),
    MetricDefinition(name='val:loss',    regex=r'\[Metrics\] Val Loss: ([0-9\.]+)'),
    MetricDefinition(name='val:top1',    regex=r'\[Metrics\] Val Top1: ([0-9\.]+)'),
    MetricDefinition(name='val:top5',    regex=r'\[Metrics\] Val Top5: ([0-9\.]+)'),
    MetricDefinition(name='val:top10',   regex=r'\[Metrics\] Val Top10: ([0-9\.]+)')
]

In [4]:
# 1. Define your code location
# source_dir='.' works because your notebook is inside Thesis-CSLR
code_config = SourceCode(
    source_dir='./ASL',
    entry_script='train_sagemaker.py' 
)

In [5]:
# 3. Hardware Configuration
# 'instance_type' and count move to this new object
compute_config = Compute(
    instance_type="ml.g5.2xlarge",
    instance_count=1
)

# Cell 2: Setup

In [6]:
# 4. Data Config
data_source = S3DataSource(
    s3_uri=f"s3://{my_bucket}/data_tensors_1000",
    s3_data_type="S3Prefix",
    s3_data_distribution_type="FullyReplicated"
)

In [7]:
data_input = InputData(
    channel_name="training",
    data_source=data_source,
    
)

In [8]:
# --- 1. Define Resume Source ---
# Point to the FOLDER (prefix) containing model.tar.gz
resume_s3_uri = "s3://alexander-thesis-cslr/experiments/output/Thesis-ASL-Exp-3DCNN-Attention-1st-1000-20260202101119/output/"

resume_source = S3DataSource(
    s3_uri=resume_s3_uri,
    s3_data_type="S3Prefix",
    s3_data_distribution_type="FullyReplicated"
)

resume_input = InputData(
    channel_name="resume",  # <--- script looks for os.environ.get("SM_CHANNEL_RESUME")
    data_source=resume_source
)

In [9]:
# 5. Output Config
output_config = OutputDataConfig(
    s3_output_path=f"s3://{my_bucket}/experiments/output"
)

In [10]:
stop_condition = StoppingCondition(
    max_runtime_in_seconds=432000
)

# Cell 3: Define the Experiment

In [11]:
# 3. Initialize the Unified Trainer
# You now provide the image URI directly
trainer = ModelTrainer(
    training_image="763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.0-gpu-py310",
    role="arn:aws:iam::600889066998:role/SageMaker_role_Alexander_for_thesis_CLSR",
    base_job_name="Thesis-ASL-Exp-3DCNN-Attention-1st_1000-RESUME",
    source_code=code_config,      # script location
    compute=compute_config,
    output_data_config=output_config,
    hyperparameters={
        "epochs": "200",
        "batch-size": "11",
        "num-classes": "1000",
        "learning-rate": "0.0001",
        "experiment-name": my_job_name,
        "model-type" : "r3d_attention",
        "start-epoch" : "94"
    },
    # Set FastFile globally for the algorithm
    training_input_mode="File",
    environment={"PYTHONUNBUFFERED": "1"},
    stopping_condition=stop_condition,
    tags=[
        {'key': 'Project', 'value': 'CSLR-Thesis'},
        {'key': 'Model',   'value': '3DCNN-Attention'},
        {'key': 'User',    'value': 'Alexander'}
    ],
    
).with_metric_definitions(metrics)


[02/04/26 11:43:02] INFO     Found credentials in environment variables.                        ]8;id=304703;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=756069;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

[02/04/26 11:43:03] INFO     SageMaker session not provided. Using default Session.                  ]8;id=905333;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=988244;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=482855;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=811164;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#162\162]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=579572;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=584380;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#548\548]8;;\
                             763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.                     
                             0-gpu-py310                                                                           

# Cell 4: Launch!

In [ ]:
print(f"🚀 Launching Job with Monitoring: {my_job_name}")
training_job = trainer.train(
    input_data_config=[data_input, resume_input],
    wait=True
)

print(f"✅ Job submitted! Check the 'Monitor' tab in the console.")

🚀 Launching Job with Monitoring: 3DCNN-Attention-Exp-1st_1000_resume


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=823875;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=782993;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[02/04/26 11:43:06] INFO     Creating training_job resource.                                     ]8;id=297929;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=595244;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=981117;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=182389;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#340\340]8;;\

...

                    INFO     Thesis-ASL-Exp-3DCNN-Attention-1st-1000-RESUME-20260204114303/algo- ]8;id=243264;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=193952;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             1-1770205435:                                                                         
                             --- Epoch 96/200 ---                                                                  

                    INFO     Thesis-ASL-Exp-3DCNN-Attention-1st-1000-RESUME-20260204114303/algo- ]8;id=823958;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=981481;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             1-1770205435:                                                                         
                             Train Ep 96:   0%|          | 0/817 [00:00<?,                                         
                             ?it/s]#033[A#033[A#015Train Ep 95:   0%|          | 0/817 [21:55<?,                   
                             ?it/s]                                                                                

[02/04/26 12:27:55] INFO     Thesis-ASL-Exp-3DCNN-Attention-1st-1000-RESUME-20260204114303/algo- ]8;id=297377;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=783639;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             1-1770205435:                                                                         
                             Train Ep: [96][204/817] Time: 284s (7.9 img/s) | Loss: 4.5865 |                       
                             Top1: 17.38% | Top5: 43.18% | Top10: 55.88%                                           

[02/04/26 12:32:37] INFO     Thesis-ASL-Exp-3DCNN-Attention-1st-1000-RESUME-20260204114303/algo- ]8;id=821926;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=772303;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             1-1770205435:                                                                         
                             Train Ep: [96][408/817] Time: 567s (7.9 img/s) | Loss: 4.5768 |                       
                             Top1: 18.18% | Top5: 43.03% | Top10: 56.15%                                           

[02/04/26 12:37:22] INFO     Thesis-ASL-Exp-3DCNN-Attention-1st-1000-RESUME-20260204114303/algo- ]8;id=336924;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=203928;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             1-1770205435:                                                                         
                             Train Ep: [96][612/817] Time: 849s (7.9 img/s) | Loss: 4.5963 |                       
                             Top1: 17.87% | Top5: 42.57% | Top10: 55.53%                                           